# Notebook 05: Spatchcocking — 3D to 2D projection

**Paper section:** Methods — Spatchcocking transformation  
**Paper figures:** Fig. 2 (coordinate system schematic), Figs. 3b/c, 4b/e, 5b/e, 6b/e (heatmaps)

This notebook performs the full spatchcocking pipeline on a mesh that already has scalar properties (curvature, thickness, pHH3 density) stored as vertex arrays.

The pipeline:
1. **Medial axis extraction** — iterative MLS smoothing along the rostral-caudal axis
2. **Cross-sectional planes** — perpendicular planes along the axis
3. **Dorsal alignment** — manual dorsal point selection to orient the azimuthal coordinate θ
4. **Thin-plate spline warping** — deforms the mesh into a cylindrical geometry
5. **Cylindrical unwrapping** — projects to 2D (θ, s) coordinates, normalizes s ∈ [0, 1]
6. **2D heatmap** — interpolates vertex values onto a regular grid for visualization

In [ ]:
from vedo import settings
settings.default_backend = "vtk"

import spatchcocking as sp
import numpy as np
import matplotlib.pyplot as plt

## Load mesh with all properties

In [ ]:
# Load a mesh that has been processed through notebooks 02–04
# (curvature + thickness + pHH3 density stored as vertex arrays)
mesh = sp.get_mesh("../data/meshes/HH17/HH17_embryo1_lumen.ply")
mesh = sp.compute_and_save_curvatures(mesh)

print("Available vertex arrays:", list(mesh.pointdata.keys()))

## Step 1: Extract medial axis

In [ ]:
# getAxis fits a 1D MLS smooth curve through the tube centroids
axis_points = sp.getAxis(mesh)
print(f"Axis: {len(axis_points)} control points")

## Step 2: Generate cross-sectional planes

In [ ]:
planes = sp.getPlanes(axis_points)
print(f"{len(planes)} planes generated along the medial axis")

## Step 3: Select dorsal points (anatomical orientation)

Dorsal points define θ = 0 (dorsal midline). In the spatchcocked projection, dorsal is at the center of the x-axis.

In [ ]:
# Interactive selection — run this cell in a VTK window
# dorsal_points = sp.selectPointsonMesh(mesh)  # click dorsal midline points

# Or load pre-saved dorsal points
# dorsal_points = np.load("HH17_embryo1_dorsal_pts.npy")

# For this demo, find dorsal points automatically (highest Y coordinate per cross-section)
dorsal_points = sp.find_closest_dorsal_points(axis_points, planes, mesh)

## Step 4: Straighten and unwrap (spatchcock)

In [ ]:
# Thin-plate spline warping into cylindrical geometry
flat_mesh = sp.getDeformedmesh(mesh, planes, axis_points, dorsal_points)

## Step 5: Extract 2D coordinates and normalize

In [ ]:
properties = ["Gauss_curvature", "Mean_curvature"]

for prop in properties:
    height, angles, values = sp.get_flatdata(flat_mesh, property=prop)
    height_norm, angles_deg, values_norm = sp.normalize_values(height, angles, values)

    fig, ax = plt.subplots(figsize=(4, 7))
    sp.visualize_flatmesh(height_norm, angles_deg, values_norm,
                          title=f"{prop} — HH17 E1", ax=ax)
    plt.tight_layout()
    plt.savefig(f"spatchcocked_{prop}.png", dpi=150)
    plt.show()

## Averaging across multiple embryos

Because all embryos are projected onto the same (θ, s) grid, data can be pooled by binning into equal-width 2D bins and computing the mean value per bin. This is how the `Spatial Average` panels (Figs. 3c, 3f, etc.) were generated.

In [ ]:
# Pseudo-code for multi-embryo pooling:

# all_heights, all_angles, all_values = [], [], []
# for mesh_path in mesh_paths:
#     mesh = sp.get_mesh(mesh_path)
#     ... (run steps 1–4) ...
#     h, a, v = sp.get_flatdata(flat_mesh, property="Gauss_curvature")
#     h, a, v = sp.normalize_values(h, a, v)
#     all_heights.append(h)
#     all_angles.append(a)
#     all_values.append(v)

# heights = np.concatenate(all_heights)
# angles  = np.concatenate(all_angles)
# values  = np.concatenate(all_values)
# sp.visualize_flatmesh(heights, angles, values, title="HH17 mean K (n=7)")